In [14]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv(r"C:\Users\micha\EXP_FARM\data\AAPL_1min.csv")
n = len(df)
truncated_df = df.loc[n*0.8:]


724083


In [10]:
def signals(data, entry_z, k=2, mom_thr=0.0, z_win=20):
    close = data["close"]

    mu  = close.rolling(z_win).mean()
    sig = close.rolling(z_win).std()
    z   = (close - mu) / sig

    mom = close / close.shift(k) - 1.0

    long_entry  = (z < -entry_z) & (mom >  mom_thr)
    short_entry = (z >  entry_z) & (mom < -mom_thr)

    return long_entry, short_entry, z, mom


In [17]:
def backtest(
    data: pd.DataFrame,
    entry_z: float,
    exit_z: float | None = None,     # if None, defaults to entry_z (other-side exit)
    k: int = 2,
    mom_thr: float = 0.0,
    z_win: int = 20,
    fee_bps: float = 0.0,
    slippage_bps: float = 0.5,
    allow_flip: bool = True,
):
    """
    Dynamic exit on opposite side of z band:
      - enter long when z < -entry_z and mom > mom_thr
      - exit long when z > +exit_z
      - enter short when z > +entry_z and mom < -mom_thr
      - exit short when z < -exit_z

    Uses next-bar execution for entries/exits (no lookahead).
    Vectorizes indicators/signals/returns/costs; uses a minimal loop for position state.
    """
    df = data.copy()
    close = df["close"].astype(float)

    if exit_z is None:
        exit_z = entry_z

    long_entry, short_entry, z, mom = signals(df, entry_z, k=k, mom_thr=mom_thr, z_win=z_win)

    # raw exit conditions (evaluated at t, acted on at t+1)
    long_exit_raw  = (z >  exit_z)
    short_exit_raw = (z < -exit_z)

    # next-bar execution
    long_entry  = long_entry.shift(1).fillna(False).to_numpy()
    short_entry = short_entry.shift(1).fillna(False).to_numpy()
    long_exit   = long_exit_raw.shift(1).fillna(False).to_numpy()
    short_exit  = short_exit_raw.shift(1).fillna(False).to_numpy()

    close_np = close.to_numpy()
    n = len(df)

    # state machine (tight numpy loop)
    pos = np.zeros(n, dtype=np.int8)      # -1, 0, 1
    tid = np.full(n, -1, dtype=np.int32)  # trade id per bar

    in_pos = 0
    trade_id = -1
    entry_px = np.nan

    trade_pnl = []
    trade_len = []

    cost_per_side = (fee_bps + slippage_bps) / 1e4

    for i in range(n):
        # 1) exits first
        if in_pos == 1 and long_exit[i]:
            # realize pnl at this bar close
            move = close_np[i] / entry_px - 1.0
            trade_pnl[trade_id] += move
            trade_pnl[trade_id] -= cost_per_side
            in_pos = 0
            entry_px = np.nan

        elif in_pos == -1 and short_exit[i]:
            move = close_np[i] / entry_px - 1.0
            trade_pnl[trade_id] += -move
            trade_pnl[trade_id] -= cost_per_side
            in_pos = 0
            entry_px = np.nan

        # 2) optional flip (close + open same bar close)
        if allow_flip and in_pos == 1 and short_entry[i]:
            # close long
            move = close_np[i] / entry_px - 1.0
            trade_pnl[trade_id] += move
            trade_pnl[trade_id] -= cost_per_side

            # open short
            in_pos = -1
            trade_id += 1
            entry_px = close_np[i]
            trade_pnl.append(-cost_per_side)
            trade_len.append(0)

        elif allow_flip and in_pos == -1 and long_entry[i]:
            # close short
            move = close_np[i] / entry_px - 1.0
            trade_pnl[trade_id] += -move
            trade_pnl[trade_id] -= cost_per_side

            # open long
            in_pos = 1
            trade_id += 1
            entry_px = close_np[i]
            trade_pnl.append(-cost_per_side)
            trade_len.append(0)

        # 3) entries if flat
        if in_pos == 0:
            if long_entry[i]:
                in_pos = 1
                trade_id += 1
                entry_px = close_np[i]
                trade_pnl.append(-cost_per_side)
                trade_len.append(0)
            elif short_entry[i]:
                in_pos = -1
                trade_id += 1
                entry_px = close_np[i]
                trade_pnl.append(-cost_per_side)
                trade_len.append(0)

        # record position and trade id
        pos[i] = in_pos
        tid[i] = trade_id if in_pos != 0 else -1
        if in_pos != 0:
            trade_len[trade_id] += 1

    # vectorized PnL series
    ret = close.pct_change().fillna(0.0)
    pos_s = pd.Series(pos, index=df.index)
    strat_gross = pos_s * ret

    # costs on position changes (0->1, 1->0, 1->-1 etc.)
    turnover = pos_s.diff().abs().fillna(0.0)
    costs = (turnover > 0).astype(float) * cost_per_side
    strat_net = strat_gross - costs

    eq = (1.0 + strat_net).cumprod()
    dd = eq / eq.cummax() - 1.0

    ann = 252 * 390
    mean = strat_net.mean()
    std = strat_net.std(ddof=0)
    sharpe = (mean / std) * np.sqrt(ann) if std > 0 else np.nan

    trade_pnl_arr = np.array(trade_pnl) if len(trade_pnl) else np.array([])
    trades = int(len(trade_pnl_arr))
    hit_rate = float(np.mean(trade_pnl_arr > 0)) if trades else np.nan
    avg_trade = float(np.mean(trade_pnl_arr)) if trades else np.nan
    avg_len = float(np.mean(trade_len)) if trades else np.nan

    out = df.copy()
    out["z"] = z
    out["mom"] = mom
    out["pos"] = pos_s
    out["ret"] = ret
    out["strat_gross"] = strat_gross
    out["costs"] = costs
    out["strat_net"] = strat_net
    out["equity"] = eq
    out["drawdown"] = dd
    out["trade_id"] = tid

    metrics = {
        "entry_z": entry_z,
        "exit_z": exit_z,
        "z_win": z_win,
        "k": k,
        "mom_thr": mom_thr,
        "fee_bps": fee_bps,
        "slippage_bps": slippage_bps,
        "allow_flip": allow_flip,
        "trades": trades,
        "hit_rate": hit_rate,
        "avg_trade_net": avg_trade,
        "avg_trade_len_bars": avg_len,
        "sharpe_approx": sharpe,
        "max_drawdown": float(dd.min()),
        "total_return": float(eq.iloc[-1] - 1.0),
    }

    return out, metrics

In [20]:
out, m = backtest(truncated_df, entry_z=2.0, exit_z=None, mom_thr=0.0002, z_win=20, slippage_bps=2)
m


C:\Users\micha\AppData\Local\Temp\ipykernel_10948\27731403.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  long_entry  = long_entry.shift(1).fillna(False).to_numpy()
C:\Users\micha\AppData\Local\Temp\ipykernel_10948\27731403.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  short_entry = short_entry.shift(1).fillna(False).to_numpy()
C:\Users\micha\AppData\Local\Temp\ipykernel_10948\27731403.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(

{'entry_z': 2.0,
 'exit_z': 2.0,
 'z_win': 20,
 'k': 2,
 'mom_thr': 0.0002,
 'fee_bps': 0.0,
 'slippage_bps': 2,
 'allow_flip': True,
 'trades': 275,
 'hit_rate': 0.5418181818181819,
 'avg_trade_net': 0.0002180806470390011,
 'avg_trade_len_bars': 54.378181818181815,
 'sharpe_approx': np.float64(0.7531227192675047),
 'max_drawdown': -0.06721748658263094,
 'total_return': 0.26703126770305463}

In [22]:
import numpy as np
import pandas as pd

def tune_grid(
    data,
    entry_z_grid=(1.5, 1.75, 2.0, 2.25, 2.5),
    exit_z_grid=None,                 # if None -> exit_z = entry_z
    z_win_grid=(15, 20, 30, 40, 60),
    k_grid=(1, 2, 3, 5),
    mom_thr_grid=(0.0, 0.0001, 0.0002, 0.0004, 0.0008),
    slippage_bps=2.0,
    fee_bps=0.0,
    allow_flip=True,
    min_trades=50,
):
    rows = []
    for z_win in z_win_grid:
        for k in k_grid:
            for mom_thr in mom_thr_grid:
                for entry_z in entry_z_grid:
                    if exit_z_grid is None:
                        exit_z_list = [None]  # uses exit_z=entry_z inside backtest
                    else:
                        exit_z_list = exit_z_grid

                    for exit_z in exit_z_list:
                        out, m = backtest(
                            data,
                            entry_z=float(entry_z),
                            exit_z=exit_z,
                            z_win=int(z_win),
                            k=int(k),
                            mom_thr=float(mom_thr),
                            slippage_bps=float(slippage_bps),
                            fee_bps=float(fee_bps),
                            allow_flip=bool(allow_flip),
                        )

                        if m["trades"] < min_trades:
                            continue

                        rows.append({
                            **m,
                            "total_return": float(m["total_return"]),
                            "sharpe_approx": float(m["sharpe_approx"]),
                            "max_drawdown": float(m["max_drawdown"]),
                            "avg_trade_net": float(m["avg_trade_net"]),
                        })

    res = pd.DataFrame(rows)
    if len(res) == 0:
        return res

    # rank: prefer higher sharpe, then higher avg_trade, then lower drawdown magnitude
    res = res.sort_values(
        by=["sharpe_approx", "avg_trade_net", "max_drawdown"],
        ascending=[False, False, False],
        kind="mergesort",
    ).reset_index(drop=True)

    return res


# run
res = tune_grid(
    truncated_df,
    slippage_bps=2.0,
    entry_z_grid=(1.5, 1.75, 2.0, 2.25, 2.5),
    z_win_grid=(15, 20, 30, 40, 60),
    k_grid=(1, 2, 3, 5),
    mom_thr_grid=(0.0, 0.0001, 0.0002, 0.0004, 0.0008),
    min_trades=80
)

res.head(20)


C:\Users\micha\AppData\Local\Temp\ipykernel_10948\27731403.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  long_entry  = long_entry.shift(1).fillna(False).to_numpy()
C:\Users\micha\AppData\Local\Temp\ipykernel_10948\27731403.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  short_entry = short_entry.shift(1).fillna(False).to_numpy()
C:\Users\micha\AppData\Local\Temp\ipykernel_10948\27731403.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(

,entry_z,exit_z,z_win,k,mom_thr,fee_bps,slippage_bps,allow_flip,trades,hit_rate,avg_trade_net,avg_trade_len_bars,sharpe_approx,max_drawdown,total_return
0,2.25,2.25,60,1,0.0001,0.0,2.0,True,3546,0.593627,0.000006,130.636210,1.664124,-0.289022,52.091951
1,2.25,2.25,60,1,0.0000,0.0,2.0,True,3955,0.592162,-0.000064,129.723894,1.653084,-0.310550,55.819565
2,2.50,2.50,60,1,0.0004,0.0,2.0,True,1450,0.596552,0.000257,166.236552,1.622317,-0.168191,8.454209
3,2.50,2.50,60,1,0.0000,0.0,2.0,True,2850,0.594035,0.000015,159.160702,1.613964,-0.292028,40.969035
4,2.50,2.50,60,1,0.0001,0.0,2.0,True,2498,0.600080,0.000109,159.866293,1.557363,-0.273942,28.726835
5,2.50,2.50,60,1,0.0002,0.0,2.0,True,2075,0.597108,0.000156,162.770602,1.533045,-0.276191,22.462279
6,2.25,2.25,60,1,0.0002,0.0,2.0,True,2978,0.595702,0.000010,131.299866,1.520230,-0.305722,28.618893
7,2.00,2.00,60,1,0.0000,0.0,2.0,True,5227,0.589822,-0.000134,106.884064,1.512444,-0.312211,43.206236
8,1.75,1.75,60,1,0.0000,0.0,2.0,True,6590,0.588467,-0.000187,89.225645,1.473670,-0.328931,41.794349
9,2.25,2.25,60,1,0.0004,0.0,2.0,True,2046,0.596285,0.000170,134.337732,1.466439,-0.191030,8.352439


In [26]:

entry_z_vals = [2.0, 2.25, 2.5]
mom_thr_vals = [0.0001, 0.0002, 0.0004]
z_win_vals   = [50, 60, 70]   # small perturbation around 60
k_vals       = [1]            # k=1 dominated
slippage_vals = [2, 3, 4, 5]  # stress test bps

rows = []

for slippage in slippage_vals:
    for entry_z in entry_z_vals:
        for mom_thr in mom_thr_vals:
            for z_win in z_win_vals:
                for k in k_vals:

                    out, m = backtest(
                        truncated_df,
                        entry_z=entry_z,
                        exit_z=entry_z,
                        z_win=z_win,
                        k=k,
                        mom_thr=mom_thr,
                        slippage_bps=slippage,
                        fee_bps=0.0,
                        allow_flip=True
                    )

                    rows.append({
                        "slippage_bps": slippage,
                        "entry_z": entry_z,
                        "z_win": z_win,
                        "mom_thr": mom_thr,
                        "trades": m["trades"],
                        "avg_trade_net": m["avg_trade_net"],
                        "sharpe": m["sharpe_approx"],
                        "max_dd": m["max_drawdown"],
                        "total_return": m["total_return"],
                    })

res = pd.DataFrame(rows)

# sort by slippage then sharpe descending
res = res.sort_values(["slippage_bps", "sharpe"], ascending=[True, False])

print(res)


C:\Users\micha\AppData\Local\Temp\ipykernel_10948\27731403.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  long_entry  = long_entry.shift(1).fillna(False).to_numpy()
C:\Users\micha\AppData\Local\Temp\ipykernel_10948\27731403.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  short_entry = short_entry.shift(1).fillna(False).to_numpy()
C:\Users\micha\AppData\Local\Temp\ipykernel_10948\27731403.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(

    slippage_bps  entry_z  z_win  mom_thr  trades  avg_trade_net    sharpe  \
10             2     2.25     60   0.0001    3546       0.000006  1.664124   
20             2     2.50     70   0.0001    2505       0.000145  1.656651   
25             2     2.50     60   0.0004    1450       0.000257  1.622317   
23             2     2.50     70   0.0002    2114       0.000196  1.570071   
19             2     2.50     60   0.0001    2498       0.000109  1.557363   
..           ...      ...    ...      ...     ...            ...       ...   
82             5     2.00     60   0.0001    4695      -0.000729  0.361232   
88             5     2.00     60   0.0004    2857      -0.000675  0.352160   
84             5     2.00     50   0.0002    4136      -0.000802  0.229582   
87             5     2.00     50   0.0004    2724      -0.000796  0.194714   
81             5     2.00     50   0.0001    4994      -0.000828  0.051328   

      max_dd  total_return  
10 -0.289022     52.091951  
20 -0

In [32]:


# ----- choose your robust params -----
PARAMS = dict(
    entry_z=2.5,
    exit_z=2.5,
    z_win=70,
    k=1,
    mom_thr=0.0001,
    fee_bps=0.0,
    slippage_bps=5.0,
    allow_flip=True
)

# ----- time split: first half train, second half test (no retune) -----
df = truncated_df.sort_index()
split_idx = len(df) // 2
train = df.iloc[:split_idx].copy()
test  = df.iloc[split_idx:].copy()

out_tr, m_tr = backtest(train, **PARAMS)
out_te, m_te = backtest(test,  **PARAMS)

print("TRAIN metrics:")
print({k: (float(v) if hasattr(v, "__float__") else v) for k, v in m_tr.items()})
print("\nTEST metrics:")
print({k: (float(v) if hasattr(v, "__float__") else v) for k, v in m_te.items()})

# optional: quick sanity on direction (mean per-bar return)
print("\nMean strat_net per bar (train/test):",
      float(out_tr["strat_net"].mean()), float(out_te["strat_net"].mean()))

# optional: compare equity endpoints
print("Total return (train/test):",
      float((1 + out_tr["strat_net"]).cumprod().iloc[-1] - 1),
      float((1 + out_te["strat_net"]).cumprod().iloc[-1] - 1))


C:\Users\micha\AppData\Local\Temp\ipykernel_10948\27731403.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  long_entry  = long_entry.shift(1).fillna(False).to_numpy()
C:\Users\micha\AppData\Local\Temp\ipykernel_10948\27731403.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  short_entry = short_entry.shift(1).fillna(False).to_numpy()
C:\Users\micha\AppData\Local\Temp\ipykernel_10948\27731403.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(

TRAIN metrics:
{'entry_z': 2.5, 'exit_z': 2.5, 'z_win': 70.0, 'k': 1.0, 'mom_thr': 0.0001, 'fee_bps': 0.0, 'slippage_bps': 5.0, 'allow_flip': 1.0, 'trades': 1253.0, 'hit_rate': 0.5395051875498803, 'avg_trade_net': -0.0004350806831877828, 'avg_trade_len_bars': 176.21308858739027, 'sharpe_approx': 0.9252544127016655, 'max_drawdown': -0.23455111549390018, 'total_return': 1.2359112513619928}

TEST metrics:
{'entry_z': 2.5, 'exit_z': 2.5, 'z_win': 70.0, 'k': 1.0, 'mom_thr': 0.0001, 'fee_bps': 0.0, 'slippage_bps': 5.0, 'allow_flip': 1.0, 'trades': 1252.0, 'hit_rate': 0.5351437699680511, 'avg_trade_net': -0.00047586480541979124, 'avg_trade_len_bars': 179.56469648562302, 'sharpe_approx': 1.260824823400693, 'max_drawdown': -0.28574875677155653, 'total_return': 4.494166840492989}

Mean strat_net per bar (train/test): 2.584970656958944e-06 5.684399723643279e-06
Total return (train/test): 1.2359112513619928 4.494166840492989
